# Lesson 3.8 — Reshaping Data with `.pivot_table()`

**Objectives**
- Reshape long-format data into a wide table with `.pivot_table()`
- Use `aggfunc` to summarize duplicate index/column combinations
- Add row and column totals with `margins=True`
- Handle missing combinations with `fill_value`
- Convert a wide table back to long format with `.melt()`


In [ ]:
import pandas as pd

In [ ]:
data_file_path = "../../data/owid-co2-data.csv"
co2_emission = pd.read_csv(data_file_path, sep=",")
selected_co2_emission = co2_emission.loc[
    co2_emission["country"].isin(
        [
            "Canada",
            "United States",
            "China",
            "Brazil",
            "Germany",
            "Russia",
            "France",
            "United Kingdom",
            "Japan",
        ]
    )
    & (co2_emission["year"] >= 2000)
    & (co2_emission["year"] <= 2024),
    ["country", "year", "co2", "gdp"],
]
selected_co2_emission.head()

## 1. `.pivot_table()`: reshape long data into a wide table

Each `(country, year)` pair appears exactly once in `selected_co2_emission`, so pivoting doesn't need to summarize anything — it's a pure reshape from long to wide.

![pivot table](./imgs/reshaping_pivot.png)

In [ ]:
co2_wide = selected_co2_emission.pivot_table(
    index="country", columns="year", values="co2"
)
co2_wide[[2020, 2021, 2022, 2023, 2024]].round(1)

In [ ]:
# set the name of the axis for the index (None), and reset index to make the country as a column
co2_wide.rename_axis(columns=None).reset_index().loc[:, ["country", 2020, 2021, 2022, 2023, 2024]]

## 2. `aggfunc` summarizes duplicate index/column combinations

Group years into decades first, so several `year` rows now map to the same `(country, decade)` cell. `.pivot_table()` needs `aggfunc` to decide how to combine them.

Apply function when pivot table and calculate the columns' values by `aggfunc`

In [ ]:
selected_co2_emission = selected_co2_emission.copy()
selected_co2_emission["decade"] = (selected_co2_emission["year"] // 10) * 10

decade_avg_co2 = selected_co2_emission.pivot_table(
    index="country", columns="decade", values="co2", aggfunc="mean"
)
decade_avg_co2.rename_axis(columns=None).reset_index().round(1)

## 3. `margins=True` adds row and column totals

In [ ]:
decade_avg_co2_with_totals = selected_co2_emission.pivot_table(
    index="country",
    columns="decade",
    values="co2",
    aggfunc="mean",
    margins=True,
    margins_name="All decades",
)
decade_avg_co2_with_totals.rename_axis(columns=None).reset_index().round(1)

The `"All decades"` row and column are computed over the *original* rows, not by averaging the cells already shown — the row total is each country's average `co2` across every year in the data, and the column total is each decade's average `co2` across every country.

## 4. `fill_value` handles missing combinations

`gdp` isn't reported yet for 2023 and 2024, so those two columns are entirely `NaN` for every country. `.pivot_table()` drops all-`NaN` columns by default (`dropna=True`), so pass `dropna=False` to keep them.

In [ ]:
gdp_wide = selected_co2_emission.pivot_table(
    index="country", columns="year", values="gdp", dropna=False
)
gdp_wide[[2021, 2022, 2023, 2024]].rename_axis(columns=None).reset_index()

In [ ]:
gdp_wide_filled = selected_co2_emission.pivot_table(
    index="country", columns="year", values="gdp", dropna=False, fill_value=0
)
gdp_wide_filled[[2021, 2022, 2023, 2024]].rename_axis(columns=None).reset_index()

`fill_value=0` replaces the remaining `NaN` cells with `0`. Use it when a missing combination genuinely means "zero", or when downstream code can't handle `NaN` — here it's mostly for illustration, since a missing GDP figure isn't really zero.

## 5. `.melt()`: wide back to long

![pivot table](./imgs/reshaping_melt.png)

In [ ]:
long_again = decade_avg_co2.reset_index().melt(
    id_vars="country", var_name="decade", value_name="avg_co2"
)
long_again.head(8)

`decade_avg_co2` was 9 countries x however many decade columns; melting turns each decade column into `(decade, avg_co2)` row pairs, giving rows = countries x decades back in long format — the shape `.groupby(["country", "decade"])["co2"].mean()` produces directly. Reshape wide when a table is for people to read; keep it long when it's headed into more pandas, plotting, or a model.